# E4 LViT-T Text Patch384 + Sliding-Window

Train text-conditioned LViT-T on Patch384 and select `best.pt` using text-conditioned full-image sliding-window validation Dice.

In [ ]:
REPO_BRANCH = 'model/lvit-patch384'
!rm -rf /kaggle/working/BTXRD-LViT
!git clone -b {REPO_BRANCH} https://github.com/lehngoc/BTXRD-LViT.git /kaggle/working/BTXRD-LViT
!pip install -q transformers

In [ ]:
import json, shutil, sys, zipfile
from pathlib import Path
import pandas as pd
import torch
import yaml

REPO_ROOT = Path('/kaggle/working/BTXRD-LViT')
target = Path('data/exports/patches_jitter/patches_384/metadata.csv')
DATA_ROOT = next((p for p in [Path('/kaggle/input'), *Path('/kaggle/input').glob('**/*')] if p.is_dir() and (p / target).exists()), None)
if DATA_ROOT is None:
    raise FileNotFoundError(f'Could not find {target} under /kaggle/input')
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('DATA_ROOT:', DATA_ROOT)

In [ ]:
required = [
    'data/exports/patches_jitter/patches_384/metadata.csv',
    'data/exports/patches_jitter_val/patches_384/metadata.csv',
    'data/exports/patches_jitter_test/patches_384/metadata.csv',
    'data/exports/btxrd_preprocessed/train.csv',
    'data/exports/btxrd_preprocessed/val.csv',
    'data/exports/btxrd_preprocessed/test.csv',
    'data/processed/images_preprocessed',
    'data/processed/masks_preprocessed',
]
missing = [path for path in required if not (DATA_ROOT / path).exists()]
if missing:
    raise FileNotFoundError('Missing required paths:\n' + '\n'.join(missing))
patch_df = pd.read_csv(DATA_ROOT / required[0])
assert 'text_lvit_prompt' in patch_df.columns
print('train patches:', len(patch_df), 'positive ratio:', patch_df.is_positive.mean())

In [ ]:
BASE_CONFIG = REPO_ROOT / 'configs/train_lvit_t_patch384.yaml'
cfg = yaml.safe_load(BASE_CONFIG.read_text())
OUTPUT_DIR = Path('/kaggle/working/experiments/E4_lvit_t_patch384_text_h4')
cfg['data']['root_dir'] = str(DATA_ROOT)
cfg['training']['device'] = 'cuda'
cfg['training']['batch_size'] = 1
cfg['training']['accumulation_steps'] = 4
cfg['training']['num_workers'] = 2
cfg['training']['output_dir'] = str(OUTPUT_DIR)
cfg['sliding_window']['batch_size'] = 1
RUNTIME_CONFIG = Path('/kaggle/working/train_lvit_t_patch384_runtime.yaml')
RUNTIME_CONFIG.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(RUNTIME_CONFIG.read_text())

In [ ]:
RESUME_FROM_INPUT = False
if RESUME_FROM_INPUT:
    candidates = list(Path('/kaggle/input').glob('**/last.pt'))
    if not candidates:
        raise FileNotFoundError('No last.pt found under /kaggle/input')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for name in ['last.pt', 'best.pt', 'history.csv', 'best_summary.json', 'config.json']:
        src = candidates[0].parent / name
        if src.exists(): shutil.copy2(src, OUTPUT_DIR / name)

In [ ]:
!python {REPO_ROOT}/src/training/smoke_test_lvit_t_patch_pipeline.py --config {RUNTIME_CONFIG}

In [ ]:
!python {REPO_ROOT}/src/training/train_lvit_t_patch.py --config {RUNTIME_CONFIG} --device cuda --auto-resume

In [ ]:
BEST_CKPT = OUTPUT_DIR / 'best.pt'
!python {REPO_ROOT}/src/inference/evaluate_lvit_t_sliding_window.py --config {RUNTIME_CONFIG} --checkpoint {BEST_CKPT} --split val --device cuda --output {OUTPUT_DIR}/val_sliding_metrics.json
!python {REPO_ROOT}/src/inference/evaluate_lvit_t_sliding_window.py --config {RUNTIME_CONFIG} --checkpoint {BEST_CKPT} --split test --device cuda --output {OUTPUT_DIR}/test_sliding_metrics.json

In [ ]:
for split in ['val', 'test']:
    metrics = json.loads((OUTPUT_DIR / f'{split}_sliding_metrics.json').read_text())
    print(split, {k: metrics[k] for k in ['tumor_dice','tumor_iou','tumor_precision','tumor_recall','normal_fp_image_rate','seconds_per_image']})

In [ ]:
RUN_VALIDATION_THRESHOLD_SWEEP = False
if RUN_VALIDATION_THRESHOLD_SWEEP:
    rows = []
    for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
        output = OUTPUT_DIR / f'val_sliding_metrics_thr{int(threshold * 100):02d}.json'
        !python {REPO_ROOT}/src/inference/evaluate_lvit_t_sliding_window.py --config {RUNTIME_CONFIG} --checkpoint {BEST_CKPT} --split val --device cuda --threshold {threshold} --output {output}
        metrics = json.loads(output.read_text())
        rows.append({k: metrics[k] for k in ['threshold','tumor_dice','tumor_iou','tumor_precision','tumor_recall','normal_fp_image_rate']})
    pd.DataFrame(rows)

In [ ]:
ARTIFACT_ZIP = Path('/kaggle/working/E4_lvit_t_patch384_text_artifacts.zip')
with zipfile.ZipFile(ARTIFACT_ZIP, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in OUTPUT_DIR.glob('*'):
        if path.is_file() and (path.suffix in {'.pt', '.json', '.csv'}): archive.write(path, path.name)
print(ARTIFACT_ZIP)